# 自动微分与工具

本notebook介绍深度学习中两个重要的实用工具:
- 自动微分(Autograd): PyTorch的自动求导系统
- API查阅: 如何查找和使用PyTorch文档

自动微分使我们无需手动计算梯度,极大简化了深度学习模型的训练。

## 第一部分: 自动微分(Autograd)

### 1.1 自动微分简介

在深度学习中,我们需要:
1. **前向传播**: 计算模型输出和损失
2. **反向传播**: 计算损失对参数的梯度
3. **参数更新**: 根据梯度更新参数

手动计算梯度既困难又容易出错。PyTorch的**自动微分系统**可以:
- 自动构建计算图
- 自动计算梯度
- 追踪所有操作以支持反向传播

In [ ]:
import torch

### 1.2 简单例子: 标量函数的梯度

计算函数$y = 2\mathbf{x}^\top\mathbf{x}$关于列向量$\mathbf{x}$的梯度。

数学推导: $\frac{\partial y}{\partial \mathbf{x}} = 4\mathbf{x}$

In [ ]:
# 创建张量并启用梯度追踪
x = torch.arange(4.0)
print("x =", x)

# 需要告诉PyTorch存储梯度
x.requires_grad_(True)
print("\n梯度初始值:", x.grad)  # None,因为还没计算

In [ ]:
# 计算 y = 2 * x^T * x
y = 2 * torch.dot(x, x)
print("y =", y)

In [ ]:
# 反向传播计算梯度
y.backward()
print("计算得到的梯度:", x.grad)
print("理论梯度 4x:", 4 * x)
print("\n验证是否相等:", x.grad == 4 * x)

### 1.3 梯度清零

**重要**: PyTorch默认会累积梯度,所以在计算新的梯度前需要清零。

In [ ]:
# 清除之前的梯度
x.grad.zero_()
print("清零后的梯度:", x.grad)

# 计算新函数 y = x.sum()
y = x.sum()
y.backward()
print("\n新的梯度:", x.grad)
print("理论值(全1):", torch.ones_like(x))

### 1.4 非标量变量的反向传播

当输出不是标量时,需要先求和再反向传播。

In [ ]:
# 清零梯度
x.grad.zero_()

# 计算向量函数 y = x * x
y = x * x
print("y =", y)
print("y的形状:", y.shape)

# 对向量求和后再反向传播
y.sum().backward()
print("\n梯度:", x.grad)
print("理论值 2x:", 2 * x)

### 1.5 分离计算

有时我们希望将某些计算移出计算图,使其被视为常数。

In [ ]:
x.grad.zero_()

# 计算 y = x * x
y = x * x

# 将y视为常数(分离计算图)
u = y.detach()
print("分离后的u:", u)

# 计算 z = u * x
z = u * x
z.sum().backward()

print("\nx的梯度:", x.grad)
print("理论值 u:", u)
print("验证:", x.grad == u)

**对比: 不分离的情况**

In [ ]:
x.grad.zero_()

# 不分离,y参与梯度计算
y = x * x
z = y * x
z.sum().backward()

print("不分离时x的梯度:", x.grad)
print("理论值 3x^2:", 3 * x * x)
print("验证:", x.grad == 3 * x * x)

### 1.6 Python控制流的梯度计算

自动微分可以处理包含控制流(if、while、for等)的函数。

In [ ]:
def f(a):
    """包含控制流的函数"""
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

In [ ]:
# 创建随机输入
a = torch.randn(size=(), requires_grad=True)
print("输入a:", a)

# 计算函数值
d = f(a)
print("输出d:", d)

# 计算梯度
d.backward()
print("\na的梯度:", a.grad)

**验证梯度**

In [ ]:
# 使用有限差分验证
# d/da ≈ (f(a+h) - f(a)) / h
a_val = a.detach()
h = 1e-4

# 重新计算(不追踪梯度)
with torch.no_grad():
    f_a = f(a_val)
    f_a_plus_h = f(a_val + h)
    numerical_grad = (f_a_plus_h - f_a) / h

print("自动微分的梯度:", a.grad)
print("数值梯度:", numerical_grad)
print("两者接近:", torch.allclose(a.grad, numerical_grad, rtol=1e-3))

### 1.7 实用技巧

**1. 禁用梯度追踪(推理时使用)**

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# 方法1: 使用上下文管理器
with torch.no_grad():
    y = x * 2
    print("y.requires_grad:", y.requires_grad)

# 方法2: 使用 @torch.no_grad() 装饰器
@torch.no_grad()
def predict(x):
    return x * 2

z = predict(x)
print("z.requires_grad:", z.requires_grad)

**2. 就地操作的注意事项**

In [ ]:
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = x * 2

# 不推荐:就地操作可能破坏计算图
# x.add_(1)  # 这会报错!

# 推荐:创建新张量
x_new = x + 1
print("安全的操作:", x_new)

**3. 查看计算图**

In [ ]:
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = x * 2
z = y.mean()

print("z的梯度函数:", z.grad_fn)
print("y的梯度函数:", y.grad_fn)
print("x是叶子节点:", x.is_leaf)
print("y是叶子节点:", y.is_leaf)

---

## 第二部分: 查阅文档

### 2.1 为什么需要查阅文档

- PyTorch有大量函数和类,无法全部记忆
- 文档包含详细的参数说明和示例
- 了解如何查文档是重要的技能

### 2.2 查找模块中的所有函数和类

使用`dir()`函数列出模块的所有属性。

In [ ]:
import torch

# 查看torch.distributions模块的所有属性
print("torch.distributions模块的内容:")
attrs = dir(torch.distributions)

# 过滤掉私有属性(以__开头的)
public_attrs = [attr for attr in attrs if not attr.startswith('_')]
print(f"\n共有{len(public_attrs)}个公开属性")
print("\n前20个:")
for attr in public_attrs[:20]:
    print(f"  - {attr}")

### 2.3 查找特定函数的用法

使用`help()`函数查看详细文档。

In [ ]:
# 查看torch.ones的文档
help(torch.ones)

### 2.4 在Jupyter中使用?查看文档

In [ ]:
# 在Jupyter中,可以使用?查看简短文档
# torch.ones?

# 使用??查看源代码
# torch.ones??

print("在Jupyter中尝试运行:")
print("  torch.ones?")
print("  torch.ones??")

### 2.5 实用的PyTorch文档查找技巧

**1. 查看张量的所有方法**

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])

# 查看张量的所有方法
tensor_methods = [m for m in dir(x) if not m.startswith('_')]
print(f"张量有{len(tensor_methods)}个公开方法/属性")
print("\n常用方法(前30个):")
for i, method in enumerate(tensor_methods[:30]):
    print(f"  {method}", end="  ")
    if (i + 1) % 5 == 0:
        print()  # 换行

**2. 查看函数签名**

In [ ]:
import inspect

# 查看函数签名
print("torch.randn的签名:")
print(inspect.signature(torch.randn))

print("\ntorch.nn.Linear的签名:")
print(inspect.signature(torch.nn.Linear))

**3. 查看对象类型和属性**

In [ ]:
x = torch.randn(3, 4)

print("对象类型:", type(x))
print("数据类型:", x.dtype)
print("设备:", x.device)
print("形状:", x.shape)
print("是否需要梯度:", x.requires_grad)

**4. 测试示例代码**

In [ ]:
# 从文档中学习后,立即测试
# 例如: torch.ones创建全1张量

# 测试不同参数
print("1D张量:")
print(torch.ones(4))

print("\n2D张量:")
print(torch.ones(2, 3))

print("\n指定数据类型:")
print(torch.ones(2, 2, dtype=torch.int32))

### 2.6 在线资源

**官方文档**:
- PyTorch官方文档: https://pytorch.org/docs/
- PyTorch教程: https://pytorch.org/tutorials/

**常用查找方式**:
1. Google搜索: "pytorch [功能名称]"
2. 查看GitHub上的示例代码
3. Stack Overflow搜索问题
4. PyTorch论坛: https://discuss.pytorch.org/

### 2.7 实战练习: 探索torch.nn模块

In [ ]:
import torch.nn as nn

# 查看nn模块中的层
nn_layers = [attr for attr in dir(nn) if not attr.startswith('_') and attr[0].isupper()]

print(f"torch.nn中有{len(nn_layers)}个类")
print("\n常用层(前20个):")
for layer in nn_layers[:20]:
    print(f"  - {layer}")

In [ ]:
# 查看Linear层的文档
print("Linear层的简要信息:")
print(f"类型: {type(nn.Linear)}")
print(f"\n签名: {inspect.signature(nn.Linear)}")

# 创建一个Linear层
layer = nn.Linear(10, 5)
print(f"\n创建的层: {layer}")
print(f"权重形状: {layer.weight.shape}")
print(f"偏置形状: {layer.bias.shape}")

---

## 小结

### 自动微分
- **自动微分**自动构建计算图并计算梯度,无需手动求导
- 使用`.requires_grad_(True)`启用梯度追踪
- 使用`.backward()`进行反向传播
- 梯度会累积,需要用`.grad.zero_()`清零
- 使用`.detach()`将张量从计算图中分离
- 使用`torch.no_grad()`上下文管理器禁用梯度(推理时)

### 文档查阅
- 使用`dir()`查看模块内容
- 使用`help()`查看详细文档
- 在Jupyter中使用`?`和`??`快速查看文档和源码
- 使用`inspect.signature()`查看函数签名
- 善用PyTorch官方文档和在线资源

## 练习

### 自动微分
1. 对函数$y = x^3 + 2x^2 - x + 1$,使用自动微分计算$x=2$时的导数,并与理论值比较
2. 创建一个2×3的矩阵$X$,计算$Y = X \cdot X^T$,然后计算$Y$关于$X$的梯度
3. 实现一个包含if-else语句的函数,验证自动微分是否正确处理

### 文档查阅
1. 查找`torch.nn.Conv2d`的文档,理解其参数含义
2. 探索`torch.optim`模块,列出所有可用的优化器
3. 查找如何保存和加载PyTorch模型的文档并实践

### 综合练习
1. 实现一个简单的线性回归模型,使用自动微分计算梯度
2. 绘制损失函数关于参数的曲线,验证梯度的正确性
3. 尝试使用`torch.autograd.grad()`函数手动计算梯度